In [ ]:

import torch
from peft import PeftModel
from unsloth import FastLanguageModel
from tqdm import tqdm

# 模型路径配置
MODEL_NAME = "/mnt/project/Qwen2.5-7B-Instruct-news/model/Qwen2.5-7B-Instruct"
LORA_PATH = "../output/mol_pretrain_epoch_2"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
model = PeftModel.from_pretrained(model, LORA_PATH)
FastLanguageModel.for_inference(model)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:

prompt = f"新闻发表时间：【2022-07-05】\n新闻标题：以“科技助企开门红”为载体实现“扛旗争先 崛起浙中”目标\n新闻内容："
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=4096,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05
)
generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
generated_content = generated.replace(prompt, "").strip()

